# Data Collection

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

##########################################
'''DATA'''
data_dir = Path("Data")
data_dir.mkdir(exist_ok=True)
##########################################
alpha = 0.3 # TODO: Find a source supporting this choice 
Tmax= 15 # above this value: no heating
Tmin = 0 #en dessous: chauffage maximal
cos_phi = 0.95
delta_Umin = 0.9 # TODO: replace with the correct value for the cable type
delta_Umax = 1.10 # TODO: replace with the correct value for the cable type
V = 20 #kV

##########################################
# 1. Load data from file 87_grid
file = data_dir / "87_0_grid.xlsx"# Load grid data
# 2. Load the area's total electricity and heat consumption profile -> converts node data into time series
npro = pd.read_excel(data_dir / "2026-05-26-Projet1-profils.xlsx",
                     sheet_name="Profils de charge",
                     skiprows=30,      # skip to row 31
                     nrows=8760)       # lire exactement 8760 lines

npro.columns = ["extra", "temps", "chaleur_kW", "chauffage_kW", "ecs_kW",
                "chaleur_centre", "froid_kW", "refroid_locaux",
                "refroid_proc", "refroid_centre", "elec_kW",
                "emob_kW", "fonctionnement_kW", "pompe_kW", "temp_C"]
# print(npro.head(3))
npro["chaleur_kW"] = pd.to_numeric(npro["chaleur_kW"], errors="coerce").fillna(0)
npro["fonctionnement_kW"] = pd.to_numeric(npro["fonctionnement_kW"], errors="coerce").fillna(0)
P_chaleur_profile = npro["chaleur_kW"].values / 1000   # MW thermique
P_elec_profile    = npro["fonctionnement_kW"].values / 1000  # electrical MW
f_load_elec    = P_elec_profile / P_elec_profile.max()    # scales P_load and Q_load
f_load_chaleur = P_chaleur_profile / P_chaleur_profile.max()  # replaces f_t for heat pumps

parameters = pd.read_excel(file, sheet_name="parameters")
f =  parameters["f_hz"].values[0] 
omega = 2 * np.pi * f

base = pd.read_excel(file, sheet_name="res_ext_grid") #TODO: REPLACE WITH NEW OPF FOR SCENARIO 2 
P_base = base["p_mw"].values
Q_base = base["q_mvar"].values

load = pd.read_excel(file, sheet_name="load")
P_load_agg = - load.groupby("bus")["p_mw"].sum() #table containing the associated load value for each node 
Q_load_agg = - load.groupby("bus")["q_mvar"].sum()

bus = pd.read_excel(file, sheet_name="bus")
n_nodes = len(bus)
nodes = range(n_nodes)

P_load = {k: float(P_load_agg.get(k, 0) or 0) for k in nodes} #Replace NaN with 0
Q_load = {k: float(Q_load_agg.get(k, 0) or 0) for k in nodes} #Replace NaN with 0
P_load_dt = np.outer(f_load_elec, [P_load[k] for k in nodes])
Q_load_dt = np.outer(f_load_elec, [Q_load[k] for k in nodes])
load_dt = []
for t in range(8760):
    for k in nodes:
        load_dt.append({
            "bus"    : k,
            "time"   : t,
            "P_load" : P_load_dt[t, k],
            "Q_load" : Q_load_dt[t, k],
        })
pd.DataFrame(load_dt).to_csv(data_dir / "load_data_dt.csv", index=False)

bus_ref = pd.read_excel(file, sheet_name="res_bus") #TODO: REPLACE WITH NEW OPF FOR SCENARIO 2 
P_ref = - bus_ref["p_mw"].values 
Q_ref = - bus_ref["q_mvar"].values

trafo = pd.read_excel(file, sheet_name="trafo")
pcc_bus = int(trafo["lv_bus"].iloc[0])

lines = pd.read_excel(file, sheet_name="line")
line_data = pd.DataFrame({
    "from_bus": lines["from_bus"],
    "to_bus": lines["to_bus"],
    "length": lines["length_km"],
    "r": lines["r_ohm_per_km"] * lines["length_km"],
    "x": lines["x_ohm_per_km"] * lines["length_km"],
    "c": lines["c_nf_per_km"] * lines["length_km"] * 1e-9,
    "I_max": lines["max_i_ka"]
})

#Construire gij, bij, bij(sh)
line_data["g"] = line_data["r"] / (line_data["r"]**2 + line_data["x"]**2)
line_data["b"] = -line_data["x"] / (line_data["r"]**2 + line_data["x"]**2)
line_data["b_sh"] = omega * line_data["c"]
#print(line_data.head()) 

#Build the Jacobian matrices
# Initialize matrices
J_Ptheta = np.zeros((n_nodes, n_nodes))
J_QU     = np.zeros((n_nodes, n_nodes))
J_PU     = np.zeros((n_nodes, n_nodes))
# Loop over each line ij
for l in range(len(line_data)):
    i = int(line_data.loc[l, "from_bus"])
    j = int(line_data.loc[l, "to_bus"])

    gij = line_data.loc[l, "g"]
    bij = line_data.loc[l, "b"]
    bsh = line_data.loc[l, "b_sh"]
# 1. MATRICE J_Ptheta
    # diagonale
    J_Ptheta[i, i] -= bij
    J_Ptheta[j, j] -= bij
    # hors diagonale
    J_Ptheta[i, j] += bij
    J_Ptheta[j, i] += bij
# 2. MATRICE J_QU
    # diagonale
    J_QU[i, i] -= (2*bsh + bij)
    J_QU[j, j] -= (2*bsh + bij)
    # hors diagonale
    J_QU[i, j] += bij
    J_QU[j, i] += bij
# 3. MATRICE J_PU
    # diagonale
    J_PU[i, i] += gij
    J_PU[j, j] += gij
    # hors diagonale
    J_PU[i, j] -= gij
    J_PU[j, i] -= gij
# 4. MATRICE J_Qtheta
J_Qtheta = - J_PU
#Calculer S_max 
line_data["S_max"] = np.sqrt(3) * line_data["I_max"] * V

##########################################
# 2. Load PV data
# The previous cell creates hourly and mean irradiance for each node.

node_irradiance_file = data_dir / "irradiance_nodes_pvlib_hourly.csv"
node_mean_file = data_dir / "irradiance_nodes_mean.csv"
expected_irradiance_year = 2023

irr_nodes = pd.read_csv(node_irradiance_file)
irr_mean = pd.read_csv(node_mean_file)
irr_nodes["datetime"] = pd.to_datetime(irr_nodes["datetime"], errors="coerce")
if irr_nodes["datetime"].isna().any() or irr_nodes["datetime"].iloc[0].year != expected_irradiance_year:
    raise ValueError(
        f"The file {node_irradiance_file} does not correspond to {expected_irradiance_year}. "
        "Run first the previous cell that creates the per-node irradiance CSV files."
    )
if len(irr_nodes) != 8760 * n_nodes:
    raise ValueError(f"The file {node_irradiance_file} must contain {8760 * n_nodes} rows, not {len(irr_nodes)}.")

duplicate_count = int(irr_nodes.duplicated(subset=["bus", "time"]).sum())
if duplicate_count:
    print(f"Duplicate bus/time entries detected in {node_irradiance_file}: {duplicate_count}. Duplicate entries were averaged.")

irradiance_by_node = (
    irr_nodes
    .pivot_table(index="time", columns="bus", values="irradiance_W_m2", aggfunc="mean")
    .reindex(index=range(8760), columns=list(nodes))
    .fillna(0)
)
G_node = irradiance_by_node.values
G_node_max = np.maximum(G_node.max(axis=0), 1e-9)
G_norm_node_dt = G_node / G_node_max

G_mean_norm_by_bus = (
    irr_mean
    .set_index("bus")["irradiance_mean_norm"]
    .reindex(list(nodes))
    .fillna(0)
)

pv_data = []    # Mean PV data per load/node
pv_data_dt = [] # Hourly PV data per load/node over the year
for k in range(n_nodes):
    Pk = abs(P_load_agg.get(k, 0))  # load au node k
    Ppv_installed = alpha * Pk      # PV power installed at node k
    irradiance_mean_factor = float(G_mean_norm_by_bus.get(k, 0))
    Ppv = Ppv_installed * irradiance_mean_factor
    Qpv = Ppv * np.tan(np.arccos(cos_phi))
    pv_data.append({
        "bus": k,
        "P_pv_installed": Ppv_installed,
        "irradiance_mean_norm": irradiance_mean_factor,
        "P_pv": Ppv,
        "Q_pv": Qpv
    })
    for t in range(8760):
        irradiance_factor_t = float(G_norm_node_dt[t, k])
        Ppv_t = Ppv_installed * irradiance_factor_t
        Qpv_t = Ppv_t * np.tan(np.arccos(cos_phi))
        pv_data_dt.append({
            "bus": k,
            "time": t,
            "irradiance_norm": irradiance_factor_t,
            "P_pv": Ppv_t,
            "Q_pv": Qpv_t
        })
pv_df = pd.DataFrame(pv_data)
P_pv_installed = pv_df.set_index("bus")["P_pv_installed"]
P_pv_max = pv_df.set_index("bus")["P_pv"]
Q_pv_max = pv_df.set_index("bus")["Q_pv"]
pv_df.to_csv(data_dir / "pv_data.csv", index=False)
pv_df_dt = pd.DataFrame(pv_data_dt)
P_pv_max_dt = pv_df_dt.set_index(["time", "bus"])["P_pv"]
Q_pv_max_dt = pv_df_dt.set_index(["time", "bus"])["Q_pv"]
pv_df_dt.to_csv(data_dir / "pv_data_dt.csv", index=False)

##########################################
# 3. Load heat-pump data
COP = 3.9         # heat pump (nPro) #TODO: update once the area is clearly defined
P_hp_total = - 26072 / 1000 # MW thermique (nPro) #TODO: update once the area is clearly defined

# temp = pd.read_csv("temperature_hourly.csv") #TODO: replace with the correct file
# T = temp["temperature_C"].values
# f_t = np.clip((Tmax - T) / (Tmax - Tmin), 0, 1) # Heating function describing heat-pump consumption as a function of outdoor temperature 

P_hp_total_elec = P_hp_total / COP #total electrical power 
Pk_total = P_load_agg.sum() # Total load

hp_data = []
hp_data_dt = []
for k in range(n_nodes):
    Pk = P_load_agg.get(k, 0)
    P_hp = P_hp_total_elec * (Pk / Pk_total)
    hp_data.append({
        "bus": k,
        "P_hp": P_hp
    })
    for t in range(len(f_load_chaleur)):
        P_hp_dt = P_hp * f_load_chaleur[t]
        hp_data_dt.append({
            "bus": k,
            "time": t,
            "P_hp": P_hp_dt
        })
hp_df = pd.DataFrame(hp_data)
P_hp_max = hp_df.set_index("bus")["P_hp"] #Structure the data for Gurobi
hp_df.to_csv(data_dir / "hp_data.csv", index=False)
hp_df_dt = pd.DataFrame(hp_data_dt)
P_hp_max_dt = hp_df_dt.set_index(["time", "bus"])["P_hp"]
hp_df_dt.to_csv(data_dir / "hp_data_dt.csv", index=False)


# Save the variables

In [3]:
import pickle

# ── Save all computed variables ──────────────────────────────
snapshot = {
    # Grid
    "n_nodes"   : n_nodes,
    "nodes"     : list(nodes),
    "pcc_bus"   : pcc_bus,
    "P_base"    : P_base,
    "Q_base"    : Q_base,
    "P_ref"     : P_ref,
    "Q_ref"     : Q_ref,
    "P_load"    : P_load,
    "Q_load"    : Q_load,
    "P_load_agg": P_load_agg,
    "Q_load_agg": Q_load_agg,
    "P_load_dt" : P_load_dt,
    "Q_load_dt" : Q_load_dt,
    "load_dt"   : load_dt,
    "Pk_total"  : Pk_total,
    "line_data" : line_data,
    "J_Ptheta"  : J_Ptheta,
    "J_QU"      : J_QU,
    "J_PU"      : J_PU,
    "J_Qtheta"  : J_Qtheta,
    # PV
    "P_pv_installed": P_pv_installed,
    "P_pv_max"    : P_pv_max,
    "Q_pv_max"    : Q_pv_max,
    "P_pv_max_dt" : P_pv_max_dt,
    "Q_pv_max_dt" : Q_pv_max_dt,
    "G_norm_node_dt": G_norm_node_dt,
    # HP
    "P_hp_max"    : P_hp_max,
    "P_hp_max_dt" : P_hp_max_dt,
    "P_hp_total" : P_hp_total,
    "COP"        : COP,
    # Profils temporels
    "P_chaleur_profile": P_chaleur_profile,
    "P_elec_profile"   : P_elec_profile,
    "f_load_elec"      : f_load_elec,
    "f_load_chaleur"   : f_load_chaleur,
    "G_norm_node_dt"    : G_norm_node_dt,
    "G_node_max"        : G_node_max,
    # Scalaires
    "alpha"    : alpha,
    "cos_phi"  : cos_phi,
    "COP"      : COP,
    "V"        : V,
    "delta_Umin": delta_Umin,
    "delta_Umax": delta_Umax,
}

data_dir = Path("Data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "ffor_data.pkl", "wb") as f_pkl:
    pickle.dump(snapshot, f_pkl)

print("✅ Data saved in Data/ffor_data.pkl")

✅ Data saved in Data/ffor_data.pkl
